In [135]:
import pandas as pd
import numpy as np

In [136]:
#Supress non-critical warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [137]:
import os
print(os.getcwd())

C:\Users\szaha\anaconda_projects\7bddeb11-5afb-41ab-a725-816dd757e48f


In [138]:
# Load 311 dataset
df_311 = pd.read_csv(r"C:\Users\szaha\Downloads\311_Service_Requests_from_2020_to_Present_20260328 (1).csv")

In [139]:
df_311.head()

,Unique Key,Created Date,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Borough,Latitude,Longitude,Location,X Coordinate (State Plane),Y Coordinate (State Plane)
0,68464973,03/26/2026 10:32:10 PM,Street Condition,Cave-in,MANHATTAN,40.748408,-73.972809,POINT (-73.972809246635 40.748407932191),"991,784","211,943"
1,68469500,03/26/2026 09:28:57 PM,Street Condition,Cave-in,BROOKLYN,40.675210,-73.961645,POINT (-73.961645217698 40.675210182368),"994,889","185,276"
2,68460323,03/26/2026 09:27:37 PM,Street Condition,Cave-in,BROOKLYN,40.675161,-73.961422,POINT (-73.961421729386 40.675160701697),"994,951","185,258"
3,68464972,03/26/2026 09:26:15 PM,Street Condition,Cave-in,STATEN ISLAND,40.576144,-74.102669,POINT (-74.102668903612 40.576144170205),"955,729","149,198"
4,68467960,03/26/2026 09:24:10 PM,Street Condition,Cave-in,STATEN ISLAND,40.570176,-74.109200,POINT (-74.109199923545 40.570176428314),"953,912","147,026"


In [140]:
df_311.columns

Index(['Unique Key', 'Created Date', 'Problem (formerly Complaint Type)',
       'Problem Detail (formerly Descriptor)', 'Borough', 'Latitude',
       'Longitude', 'Location', 'X Coordinate (State Plane)',
       'Y Coordinate (State Plane)'],
      dtype='object')

In [141]:
df_311.shape

(55968, 10)

In [142]:
# drop rows that do not contain latitude or longitude
df_311 = df_311.dropna(subset=["Latitude", "Longitude"])

In [143]:
# Convert 'Created Date' to datetime format
df_311["Created Date"] = pd.to_datetime(df_311["Created Date"])

In [144]:
# Extract year, month, and year-month
df_311["year"] = df_311["Created Date"].dt.year
df_311["month"] = df_311["Created Date"].dt.month

df_311["year_month"] = df_311["Created Date"].dt.to_period("M")

In [145]:
# remove duplicates
df_311 = df_311.drop_duplicates(subset="Unique Key")

In [146]:
df_311.shape

(53056, 13)

In [147]:
# upload NOAA precipitation data
df_precip = pd.read_csv(r"C:\Users\szaha\Downloads\precipitation_data_.csv")

In [148]:
#Convert precipitation "DATE" column to datetime format and derive year, month, and year-month fields
df_precip['DATE'] = pd.to_datetime(df_precip['DATE'])

In [149]:
# create year, month, and Year_Month
df_precip['Year'] = df_precip['DATE'].dt.year
df_precip['Month'] = df_precip['DATE'].dt.month
df_precip['Year_Month'] = df_precip['DATE'].dt.to_period('M')

In [150]:
#Check for duplicates
print(df_precip.duplicated().sum())

0


In [151]:
#Aggregate daily precipitation into monthly totals and counts
df_precip_monthly = df_precip.groupby('Year_Month').agg({
    'PRCP':'sum',
    'DATE':'count'
}).reset_index()

In [152]:
# rename columns
df_precip_monthly.columns = ['Year_Month', 'Monthly_Precip', 'Days_Recorded']

In [153]:
df_precip_monthly.columns = ['Year_Month', 'Monthly_Precip', 'Days_Recorded']

In [154]:
# restrict datasets to study period (2020-01 to 2025-11)
start = "2020-01"
end = "2025-11"

In [155]:
df_311 = df_311[
    (df_311["year_month"] >= start) &
    (df_311["year_month"] <= end)
]

In [156]:
df_precip_monthly = df_precip_monthly[
    (df_precip_monthly["Year_Month"] >= start) &
    (df_precip_monthly["Year_Month"] <= end)
]

In [157]:
print(df_311["year_month"].min(), df_311["year_month"].max())
print(df_precip_monthly["Year_Month"].min(), df_precip_monthly["Year_Month"].max())

2020-01 2025-11
2020-01 2025-11


In [158]:
df_precip.to_csv("cleaned_precipitation.csv", index=False)

In [159]:
df_311.to_csv("cleaned_311_points.csv", index=False)

In [160]:
df_water = pd.read_csv(r"C:\Users\szaha\Downloads\WATER_MAIN_BREAKS_SINCE_jan_1_2010_20260328 (3).csv")

In [161]:
df_water.head()

,Unique Key,Created Date,Borough,X Coordinate (State Plane),Y Coordinate (State Plane),Latitude,Longitude,Location,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor)
0,45285914,2020 Jan 01 07:27:00 PM,BROOKLYN,"1,001,813","157,448",40.598817,-73.936756,POINT (-73.936755863966 40.598817250138),Water System,Possible Water Main Break (Use Comments) (WA1)
1,45288631,2020 Jan 01 07:42:00 PM,QUEENS,"1,031,784","193,484",40.697618,-73.828577,POINT (-73.828576925532 40.697617994527),Water System,Possible Water Main Break (Use Comments) (WA1)
2,45286301,2020 Jan 01 11:11:00 PM,BROOKLYN,"1,001,805","157,444",40.598806,-73.936785,POINT (-73.936784682248 40.5988062868),Water System,Possible Water Main Break (Use Comments) (WA1)
3,45293371,2020 Jan 02 07:23:00 AM,BROOKLYN,"1,010,541","175,595",40.648605,-73.905256,POINT (-73.905255790488 40.648605343463),Water System,Possible Water Main Break (Use Comments) (WA1)
4,45290644,2020 Jan 02 08:23:00 AM,BROOKLYN,"1,011,606","171,560",40.637527,-73.901434,POINT (-73.901434241749 40.637526917178),Water System,Possible Water Main Break (Use Comments) (WA1)


In [162]:
df_water.shape

(24718, 10)

In [163]:
df_water = df_water.dropna(subset=["Latitude", "Longitude"])

In [164]:
df_water['Created Date'] = pd.to_datetime(df_water['Created Date'])

In [165]:
df_water['Year'] = df_water['Created Date'].dt.year
df_water['Month'] = df_water['Created Date'].dt.month
df_water['Year_Month'] = df_water['Created Date'].dt.to_period('M')

In [166]:
print(df_water.duplicated().sum())

0


In [167]:
df_water = df_water[
    (df_water["Year_Month"] >= start) &
    (df_water["Year_Month"] <= end)
]

In [169]:
df_water.to_csv("cleaned_water.csv", index=False)